Este notebook tiene como finalidad desarrollar una solución automatizada que permita identificar posibles conflictos de segregación de funciones (Segregation of Duties, SoD) dentro de procesos de negocio en un ERP real y extendido como SAP. La segregación de funciones constituye un control fundamental de gobierno, riesgo y cumplimiento, ya que busca la concentración incompatible de funciones, es decir, evitar que una misma persona posea privilegios incompatibles que puedan facilitar errores, fraudes o acciones no autorizadas.

In [1]:
import pandas as pd
import numpy as np
from itertools import combinations

##### enfoque sencillo

In [2]:
roles_df = pd.DataFrame([
    ["U001", "Juan Pérez", "CREATE_VENDOR"],
    ["U001", "Juan Pérez", "PROCESS_PAYMENT"],
    ["U002", "Ana Ruiz", "CREATE_PO"],
    ["U002", "Ana Ruiz", "APPROVE_PO"],
    ["U003", "Carlos López", "CREATE_PO"],
    ["U003", "Carlos López", "VIEW_REPORTS"],
    ["U004", "María Gómez", "CREATE_VENDOR"],
    ["U004", "María Gómez", "VIEW_REPORTS"],
    ["U004", "María Gómez", "PROCESS_PAYMENT"],

], columns=[
    "user_id",
    "user_name",
    "role"
])

roles_df

,user_id,user_name,role
0,U001,Juan Pérez,CREATE_VENDOR
1,U001,Juan Pérez,PROCESS_PAYMENT
2,U002,Ana Ruiz,CREATE_PO
3,U002,Ana Ruiz,APPROVE_PO
4,U003,Carlos López,CREATE_PO
5,U003,Carlos López,VIEW_REPORTS
6,U004,María Gómez,CREATE_VENDOR
7,U004,María Gómez,VIEW_REPORTS
8,U004,María Gómez,PROCESS_PAYMENT


In [3]:
sod_rules = pd.DataFrame([
    ["CREATE_VENDOR", "PROCESS_PAYMENT", "CRITICAL"],
    ["CREATE_PO", "APPROVE_PO", "HIGH"]
], columns=[
    "role_a",
    "role_b",
    "risk_level"
])

sod_rules

,role_a,role_b,risk_level
0,CREATE_VENDOR,PROCESS_PAYMENT,CRITICAL
1,CREATE_PO,APPROVE_PO,HIGH


In [4]:
violations = []

for user, group in roles_df.groupby("user_id"):

    roles = set(group["role"])

    for role1, role2 in combinations(roles, 2):

        match = sod_rules[
            (
                (sod_rules["role_a"] == role1)
                & (sod_rules["role_b"] == role2)
            )
            |
            (
                (sod_rules["role_a"] == role2)
                & (sod_rules["role_b"] == role1)
            )
        ]

        if not match.empty:

            violations.append({
                "user_id": user,
                "role_1": role1,
                "role_2": role2,
                "risk": match.iloc[0]["risk_level"]
            })

violations_df = pd.DataFrame(violations)

violations_df

,user_id,role_1,role_2,risk
0,U001,CREATE_VENDOR,PROCESS_PAYMENT,CRITICAL
1,U002,CREATE_PO,APPROVE_PO,HIGH
2,U004,CREATE_VENDOR,PROCESS_PAYMENT,CRITICAL


##### ejemplo usando: Usuario -> Rol -> Transacción, tal como se hace con un ERP real como SAP

In [ ]:
# creación de una tabla con userid y rol
user_roles = pd.DataFrame([
    ["U001", "Juan", "AP_CLERK"],
    ["U001", "Juan", "AP_MANAGER"],
    ["U002", "Ana", "BUYER"],
    ["U002", "Ana", "PO_APPROVER"],
    ["U003", "Carlos", "BUYER"],
    ["U004", "María", "GL_ACCOUNTANT"],
    ["U004", "María", "GL_MANAGER"]

], columns=[
    "user_id",
    "user_name",
    "role"
])

user_roles

,user_id,user_name,role
0,U001,Juan,AP_CLERK
1,U001,Juan,AP_MANAGER
2,U002,Ana,BUYER
3,U002,Ana,PO_APPROVER
4,U003,Carlos,BUYER
5,U004,María,GL_ACCOUNTANT
6,U004,María,GL_MANAGER


In [ ]:
# creación de una tabla con rol y transacción
role_transactions = pd.DataFrame([
    ["AP_CLERK", "FK01"],
    ["AP_CLERK", "FK02"],
    ["AP_MANAGER", "F110"],
    ["AP_MANAGER", "FB60"],
    ["BUYER", "ME21N"],
    ["PO_APPROVER", "ME29N"],
    ["GL_ACCOUNTANT", "FB50"],
    ["GL_MANAGER", "FBV0"]

], columns=[
    "role",
    "transaction"
])

role_transactions

,role,transaction
0,AP_CLERK,FK01
1,AP_CLERK,FK02
2,AP_MANAGER,F110
3,AP_MANAGER,FB60
4,BUYER,ME21N
5,PO_APPROVER,ME29N
6,GL_ACCOUNTANT,FB50
7,GL_MANAGER,FBV0


In [7]:
# donde:
# FK01 = Crear proveedor
# F110 = Ejecutar pagos
# ME21N = Crear orden compra
# ME29N = Aprobar orden compra
# FB50 = Crear asiento manual
# FBV0 = Aprobar asiento

In [ ]:
# bajo este enfoque realista el conflicto ya no es por rol, es por transacción
sod_matrix = pd.DataFrame([
    [
        "FK01",
        "F110",
        "CRITICAL",
        "Crear proveedor y ejecutar pagos"
    ],
    [
        "ME21N",
        "ME29N",
        "HIGH",
        "Crear y aprobar OC"
    ],
    [
        "FB50",
        "FBV0",
        "CRITICAL",
        "Crear y aprobar asiento"
    ]

], columns=[
    "trx_a",
    "trx_b",
    "risk",
    "description"
])

sod_matrix

,trx_a,trx_b,risk,description
0,FK01,F110,CRITICAL,Crear proveedor y ejecutar pagos
1,ME21N,ME29N,HIGH,Crear y aprobar OC
2,FB50,FBV0,CRITICAL,Crear y aprobar asiento


In [9]:
# expandir usuario + transacción
user_transactions = (
    user_roles
    .merge(
        role_transactions,
        on="role",
        how="left"
    )
)

user_transactions

,user_id,user_name,role,transaction
0,U001,Juan,AP_CLERK,FK01
1,U001,Juan,AP_CLERK,FK02
2,U001,Juan,AP_MANAGER,F110
3,U001,Juan,AP_MANAGER,FB60
4,U002,Ana,BUYER,ME21N
5,U002,Ana,PO_APPROVER,ME29N
6,U003,Carlos,BUYER,ME21N
7,U004,María,GL_ACCOUNTANT,FB50
8,U004,María,GL_MANAGER,FBV0


In [10]:
# detección de casos SoD

violations = []

for user_id, group in user_transactions.groupby("user_id"):
    user_name = group["user_name"].iloc[0]
    trx_set = set(group["transaction"])

    for trx1, trx2 in combinations(trx_set, 2):
        conflict = sod_matrix[
            (
                (sod_matrix["trx_a"] == trx1)
                & (sod_matrix["trx_b"] == trx2)
            )
            |
            (
                (sod_matrix["trx_a"] == trx2)
                & (sod_matrix["trx_b"] == trx1)
            )
        ]

        if not conflict.empty:
            violations.append({
                "user_id": user_id,
                "user_name": user_name,
                "transaction_1": trx1,
                "transaction_2": trx2,
                "risk": conflict.iloc[0]["risk"],
                "description": conflict.iloc[0]["description"]
            })

violations_df = pd.DataFrame(violations)
violations_df

,user_id,user_name,transaction_1,transaction_2,risk,description
0,U001,Juan,FK01,F110,CRITICAL,Crear proveedor y ejecutar pagos
1,U002,Ana,ME21N,ME29N,HIGH,Crear y aprobar OC
2,U004,María,FBV0,FB50,CRITICAL,Crear y aprobar asiento


el siguiente paso natural sería encontrar en un log: user_activity, los conflictos que se han concretado

### Conclusión

Python es muy útil en la creación de un script para la detección continua de problemas relacionados con SoD, como una iniciativa para fortalecer los controles internos, optimizar las actividades de auditoría y cumplimiento, reducir riesgos operativos y proporcionar un mecanismo automatizado y escalable para la identificación temprana de conflictos de acceso y responsabilidades dentro de la organización. Un script muy similar a este detectó 50 casos de incompatibilidad en una revisión normal de P2P. 